# Julia notebook to compute the symbolic solution to the dissipative buoyancy equation
twnh 

Dec '25   Write ExampleSolution_v0.5.ipynb <br>
Jun '26.  Begin BuoyancyEquationSolution.ipynb

This notebook solves the non-dimensional, linear, time-independent buoyany equation with relaxation:
\begin{align}
Ek \frac{d}{dz} \left( \kappa \frac{d b}{d z} \right) + \Gamma ( B(z) - b ) = 0
\end{align}
for $b(z)$, given diffusivity $\kappa = \kappa (z)$, and boundary conditions
\begin{align}
\text{Surface~}z = 0: b &= b_s ,\\
\text{Bottom~}z = -H: \frac{\partial b}{\partial z} &= 0
\end{align}
where $B(z)$ is the specified relaxation buoyancy field, $b_s$ is the specified surface buoyancy field, and $\Gamma$ is the relaxation rate.

In [1]:
using SymPy
im = SymPy.im  # SymPy's imaginary unit
notebook_name = "BuoyancyEquationSolution"
# using Infiltrator
using Symbolics
const I = im    # Define I as the imaginary unit
# using Gridap
# using GridapGmsh
# using GridapMakie
# using Gridap.Geometry
# using Gridap.ReferenceFEs
# using WriteVTK
# using Profile
# using ProfileView

im

#### Function to convert a `SymPy` expression to `Symbolics`. This replaces `lambdify` calls (slow). However, for some complicated `SymPy` expressions, it's too slow to make this conversion.

In [2]:
function build_Symbolics_fn(expr,varnames)
	vars = [Symbolics.Variable(Symbol(n)) for n in varnames]							# Extract Symbolics variables from variable names
	tmp1 = sympy_to_symbolics(expr, vars)			
	I_vars = filter(v -> string(v) == "I", Symbolics.get_variables(tmp1))				# Check if imaginary unit variable is present and substitute
	if !isempty(I_vars)
    	tmp2 = substitute(tmp1, Dict(first(I_vars) => im))
	else
	    tmp2 = tmp1
	end
	tmp3vec = build_function(tmp2, vars; expression=Val{false})

	# Wrap function to throw an error if it's called with multiple scalar arguments rather than a vector:
	function tmp3(args::AbstractVector)
    	length(args) == length(vars) || throw(ArgumentError("Input must be vector of length $(length(vars))."))
    	tmp3vec(args)
	end
	tmp3(args...) = throw(MethodError(tmp3, args))

	return tmp3
end

build_Symbolics_fn (generic function with 1 method)

### Define symbols:

In [3]:
# Geometry symbolic parameters:
z, ξ   = symbols("z ξ",   real=true, negative=true) # Vertical coordinate and source location (both in [-H,0])
x, y   = symbols("x y",   real=true)                # Horizontal coordinates
H      = SymFunction("H", real=true, positive=true) # Domain depth H(x,y)
geometry_params = (H(x,y), z, ξ)

Ek     = symbols("Ek ",   real=true, positive=true) # Ekman number
ϕ      = symbols("ϕ",     real=true, positive=true) # Viscosity parameters

# Buoyancy equation symbolic parameters:
b      = SymFunction("b", real=true)                # Buoyancy field b(x,y,z)
ψ      = symbols("ψ",     real=true, positive=true) # Diffusivity parameters
Γ, α   = symbols("Γ α",   real=true, positive=true) # Relaxation parameter
B      = SymFunction("B", real=true)                # Relaxation buoyancy field

# Define diffusivity profile here:
κ = 1.0 + 0.0* z                                    # Constant viscosity profile. Aspire to depth-varying profile in future.

b_params = (κ, Ek , Γ) 

# Convenience variables:
root_iϕ = ϕ*sympy.sqrt(im)
sample_params = Dict(Ek =>2, ψ=>3, root_iϕ=>4im, ϕ => 4, H(x,y)=>1 - x^2 - y^2)
# sample_point  = Dict(x=>-0.6, y=>0.2, z=>-0.32)

Dict{Sym{PyCall.PyObject}, Number} with 5 entries:
  Ek        => 2
  sqrt(I)*ϕ => 0+4im
  ϕ         => 4
  H(x, y)   => -x^2 - y^2 + 1
  ψ         => 3

#### Function to solve the buoyancy equation using a Green's function:

In [4]:
function compute_Gb(b_params, geometry_params)
    # Setup symbols and parameters:
    κexpr, Ek , Γ = b_params
    H, z, ξ = geometry_params
    b       = SymFunction("b")
    A       = symbols("A",     real=true)                      # Unknown coefficient in the Green's function solution
    
    #0. Define the ODE for b(z):
    ode = Eq( - Γ * b(z)     + Ek * diff(κexpr *  diff(b(z),z),z), 0)

    # 1. Solve for $G_- (z)$ on $-H \le z \le ξ$:
    bm = dsolve(ode, b(z), ics = Dict(diff(b(z),z).subs(z,-H)=>0)).rhs
    @assert sympy.simplify(ode.lhs.subs(b(z),bm)) == 0                  # Check solution
    @assert sympy.simplify(diff(bm,z).subs(z,-H) - 0) == 0              # Check Neumann BC at bottom
    bm_const = filter(x -> startswith(string(x), "C"), bm.free_symbols)
    bm = bm.subs(first(bm_const), A)                              # Replace constant with A so it doesn't conflict later

    # 2. Solve for $G_+(z)$ on  $ξ \le z \le 0$:
    bp = dsolve(ode, b(z), ics = Dict(b(0)=>0) ).rhs
    @assert sympy.simplify(ode.lhs.subs(b(z),bp)) == 0                  # Check solution
    @assert sympy.simplify(bp.subs(z,0)) == 0                           # Check Dirichlet BC at top

    # 3. Compute Wronskian $W(z)$:
    W = sympy.simplify(bm * diff(bp, z) - bp * diff(bm, z))

    # 4. Compute Green's function $G(z; ξ)$:
    Gm = sympy.simplify(bm * bp.subs(z,ξ) / (Ek * κexpr.subs(z,ξ) * W.subs(z,ξ)))
    Gp = sympy.simplify(bm.subs(z,ξ) * bp / (Ek * κexpr.subs(z,ξ) * W.subs(z,ξ)))

    # Check continuity and jump condition:
    @assert Gm.subs(z,ξ) - Gp.subs(z,ξ) == 0
    @assert sympy.simplify(diff(Gm, z).subs(z,ξ) - diff(Gp, z).subs(z,ξ)) + 1/(Ek * κexpr.subs(z,ξ)) == 0

    # Define piecewise Green's function:
    G = sympy.simplify(sympy.Piecewise((Gm, Le(z,ξ)), (Gp, Ge(z,ξ))))

    # Check boundary conditions are satisfied
    @assert sympy.simplify(diff(G,z).subs(z,-H).subs(ξ,-H//3).subs(H,1//2)) == 0
    @assert sympy.simplify(G.subs(z,0).subs(ξ,-H//3)) == 0

    return G
end

compute_Gb (generic function with 1 method)

### Compute the buoyancy Green's function:

In [5]:
Gb_sym  = compute_Gb(b_params,geometry_params)
Gb = Gb_sym.subs(Γ,ψ^2 * Ek)

⎧    ⎛     -2⋅ξ⋅ψ⎞ ⎛ 2⋅ψ⋅H(x, y)    -2⋅z⋅ψ⎞  ψ⋅(z + ξ)           
⎪0.5⋅⎝1 - ℯ      ⎠⋅⎝ℯ            + ℯ      ⎠⋅ℯ                    
⎪─────────────────────────────────────────────────────  for z ≤ ξ
⎪                    ⎛ 2⋅ψ⋅H(x, y)    ⎞                          
⎪               Ek⋅ψ⋅⎝ℯ            + 1⎠                          
⎨                                                                
⎪    ⎛     -2⋅z⋅ψ⎞ ⎛ 2⋅ψ⋅H(x, y)    -2⋅ξ⋅ψ⎞  ψ⋅(z + ξ)           
⎪0.5⋅⎝1 - ℯ      ⎠⋅⎝ℯ            + ℯ      ⎠⋅ℯ                    
⎪─────────────────────────────────────────────────────  for z ≥ ξ
⎪                    ⎛ 2⋅ψ⋅H(x, y)    ⎞                          
⎩               Ek⋅ψ⋅⎝ℯ            + 1⎠                          

#### Compute the buoyancy term explicitly for a linear relaxation buoyancy profile. This is to check and support the Latex derivation in `PlanetaryGeostrophySolutions_V0.5.tex`. The symbolic expressions themselves are used in `ExampleSolution_v0.6.ipynb`, which avoid SymPy by using `Symbolics.jl` instead. `Symbolics.jl` can't manage the integrations done by `Sympy` in this notebook.

In [6]:
# Compute $z$ integral of Gb for Latex notes:
max_obj = sympy.Max(z, -H(x, y))
Gb_int = SymPy.integrate(Gb,(z,z,0))
Gb_int = Gb_int.subs(max_obj, z)
Gb_int = sympy.simplify(Gb_int)
Latex_Gb_int_den = 2 * ψ^2 * Ek * (exp(2 * ψ * H(x,y)) + 1)
Latex_Gb_int_num = 
    exp(   ψ * ( z - ξ + 2 * H(x,y))) -
    exp(   ψ * ( z + ξ + 2 * H(x,y))) +
2 * exp(   ψ * ( ξ + 2 * H(x,y) ) ) + 
2 * exp( - ψ * ξ) +  
    exp(   ψ * ( ξ - z )) - 
    exp( - ψ * ( z + ξ )) -
    exp(   ψ * ( ξ + 2 * H(x,y) -  sympy.Max( z, ξ ))) - 
    exp( - ψ * ( ξ - 2 * H(x,y) -  sympy.Max( z, ξ ) )) -
    exp(   ψ * ( ξ - sympy.Max( z, ξ ) )) - 
    exp( - ψ * ( ξ - sympy.Max( z, ξ ) )) 

@assert sympy.simplify(Latex_Gb_int_num / Latex_Gb_int_den - Gb_int) == 0
println("1.1 LaTeX Gb integral matches.")

# Compute integrand for linear relaxation buoyancy profile explicitly:
tmp_integrand = sympy.expand(- Gb_int * ξ * α * Γ)
tmp_integral  = sympy.integrate(tmp_integrand, (ξ, -H(x,y), 0))
tmp_integral  = sympy.simplify(tmp_integral.subs(max_obj, z))
Latex_Gb_int_int = z^2 +
    2*( 2 * exp(ψ * H(x,y)) - exp(ψ * (z + H(x,y))) - exp(ψ * (H(x,y) - z)) ) / (ψ^2 * (exp(2 * ψ * H(x,y)) + 1) )
Latex_Gb_int_int = (- α * Γ) * Latex_Gb_int_int / (2 * Ek * ψ^2)
@assert sympy.simplify(Latex_Gb_int_int  - tmp_integral) == 0
println("1.2 LaTeX Gb double integral with linear buoyancy relaxation profile matches.")

# Differentiate Latex_Gb_int_int to get the baroclinic pressure gradient:
dGb_int_int_∂z = sympy.simplify(diff(Latex_Gb_int_int,x) + im*diff(Latex_Gb_int_int,y))
Latex_dGb_int_int_∂z = 
( ( (exp(ψ * H(x,y)) - exp(3 * ψ * H(x,y))) * (exp(ψ * z) - 2 + exp(- ψ * z)) ) / ( exp(2 * ψ * H(x,y)) + 1 )^2 ) * (diff(H(x,y),x) + im * diff(H(x,y),y))
Latex_dGb_int_int_∂z = (α * Γ) * Latex_dGb_int_int_∂z / (Ek * ψ^3)
@assert sympy.simplify(dGb_int_int_∂z - Latex_dGb_int_int_∂z) == 0
println("1.3 LaTeX vertical integral of horizontal buoyancy derivative with linear buoyancy relaxation profile matches.")

# Construct the buoyancy related T(x) term (excludes the stress-driven part):
Txb_term_integrand = exp(-root_iϕ * (ξ - H(x,y))) * (1 + exp(2*root_iϕ * ξ)) / (1 + exp(2*root_iϕ * H(x,y)))
Txb_term_integrand = Txb_term_integrand * dGb_int_int_∂z.subs(z,ξ)
Txb_term = sympy.integrate(sympy.expand(Txb_term_integrand), (ξ, -H(x,y), 0))
Txb_term = sum(Txb_term.args[i] for i in 1:length(Txb_term.args)-1) + Txb_term.args[length(Txb_term.args)].args[1].args[1]    # Manually expand the integral result

Latex_Txb_term = α * Γ * (exp((ψ + root_iϕ) * H(x,y)) - exp((3*ψ + root_iϕ) * H(x,y)))/
(Ek * ψ^3 * (1 + exp(2 * ψ * H(x,y)))^2 * (1 + exp(2 * root_iϕ * H(x,y)) ))
Latex_Txb_term = Latex_Txb_term * (sympy.diff(H(x,y),x) + im*sympy.diff(H(x,y),y))
Latex_int_result = (
((1 - exp(- (ψ - root_iϕ)*H(x,y)))/( ψ - root_iϕ)) +
((1 - exp(- (ψ + root_iϕ)*H(x,y)))/( ψ + root_iϕ)) +
(2/root_iϕ) * (exp(-root_iϕ * H(x,y)) - exp(root_iϕ * H(x,y))) +
((1 - exp(  (ψ + root_iϕ)*H(x,y)))/(-ψ - root_iϕ)) + 
((1 - exp(  (ψ - root_iϕ)*H(x,y)))/(-ψ + root_iϕ)) 
)
# Check intermediate analytic Latex integral:
inter1 = exp(- (ψ - root_iϕ)*ξ) + exp(- (ψ + root_iϕ)*ξ) - 2*(exp(-root_iϕ*ξ) + exp(+root_iϕ*ξ)) + exp((ψ + root_iϕ)*ξ) + exp((ψ - root_iϕ)*ξ)
inter2 = sympy.integrate(inter1, (ξ,-H(x,y),0))
@assert inter2 - Latex_int_result == 0
Latex_Txb_term = Latex_Txb_term * Latex_int_result
# Substitute specific parameter values to check equality, otherwise SymPy takes a very long time...
tmp = (sympy.expand(Latex_Txb_term) - sympy.expand(Txb_term)).subs(sample_params).doit()  # For easier comparison
tmp = sympy.simplify(tmp)
@assert tmp == 0
println("1.4 LaTeX T(x) buoyancy term with linear buoyancy relaxation profile matches.")

# Construct the buoyancy related B(x) term (excludes the stress-driven part):
Bxb_term_integrand = exp(-root_iϕ * ξ) * (exp(root_iϕ * ξ) - exp(root_iϕ * H(x,y))) * (exp(root_iϕ * (ξ + H(x,y))) - 1)
Bxb_term_integrand = Bxb_term_integrand * dGb_int_int_∂z.subs(z,ξ)
Bxb_term = sympy.integrate(sympy.expand(Bxb_term_integrand), (ξ, -H(x,y), 0))

Latex_Bxb_term = α * Γ * (exp(ψ * H(x,y)) - exp(3 * ψ * H(x,y)))/(Ek * ψ^3 * (1 + exp(2 * ψ * H(x,y)))^2)
Latex_Bxb_term = Latex_Bxb_term * (sympy.diff(H(x,y),x) + im*sympy.diff(H(x,y),y))
Latex_int_result = exp(root_iϕ * H(x,y)) * (
((1 - exp(- (ψ - root_iϕ)*H(x,y)))/( ψ - root_iϕ)) +
((1 - exp(- (ψ + root_iϕ)*H(x,y)))/( ψ + root_iϕ)) +
(2/root_iϕ) * (exp(-root_iϕ * H(x,y)) - exp(root_iϕ * H(x,y))) +
((1 - exp(  (ψ + root_iϕ)*H(x,y)))/(-ψ - root_iϕ)) + 
((1 - exp(  (ψ - root_iϕ)*H(x,y)))/(-ψ + root_iϕ)) +
((exp((-ψ + root_iϕ) * H(x,y)) - exp((ψ + root_iϕ) * H(x,y)))/ψ)
)
Latex_int_result = Latex_int_result + (1/ψ)*(exp(-ψ * H(x,y)) - exp(ψ * H(x,y))) + 2*H(x,y)*(1 + exp(2*root_iϕ * H(x,y)))
# Check intermediate analytic Latex integral:
inter1 = (1 - exp(root_iϕ * (H(x,y) - ξ))) * (exp(root_iϕ * (H(x,y) + ξ)) - 1) * (exp(ψ * ξ) - 2 + exp(- ψ * ξ))
inter2 = sympy.integrate(sympy.expand(inter1), (ξ,-H(x,y),0))
@assert inter2 - Latex_int_result == 0
Latex_Bxb_term = Latex_Bxb_term * Latex_int_result
# Substitute specific parameter values to check equality, otherwise SymPy takes a very long time...
tmp = (sympy.expand(Latex_Bxb_term) - Bxb_term).subs(sample_params).doit()  # For easier comparison
tmp = sympy.simplify(tmp)
@assert tmp == 0
println("1.5 LaTeX B(x) buoyancy term with linear buoyancy relaxation profile matches.")

1.1 LaTeX Gb integral matches.
1.2 LaTeX Gb double integral with linear buoyancy relaxation profile matches.
1.3 LaTeX vertical integral of horizontal buoyancy derivative with linear buoyancy relaxation profile matches.
1.4 LaTeX T(x) buoyancy term with linear buoyancy relaxation profile matches.
1.5 LaTeX B(x) buoyancy term with linear buoyancy relaxation profile matches.


### Symbolic computation of buoyancy $b$

In [7]:
# Compute buoyancy field b(z):
b_sym = SymPy.integrate(sympy.expand(-Gb * Γ * B(x,y,ξ)), (ξ,-H(x,y),0) )
b = sympy.simplify(b_sym.subs(max_obj,z))

# Check that the final expression for b satisfies the original differential equation:
tmp1 = Ek * diff(diff(κ * b,z),z) - Γ * b
tmp1 = sympy.simplify(tmp1.subs(ψ,sqrt(Γ/Ek) ))
@assert tmp1 + Γ * B(x,y,z) == 0

In [10]:
# Define the parameter values
f_val  = 1
Ek_val = 0.95
Γ_val  = 0.2

# Compute compound parameter:
ψ_val = sqrt(Γ_val / Ek_val)

# Define domain depth:
H_val(x,y)    = 1.0 - x^2 - y^2

# Define relaxation buoyancy field:
B_val(x,y,ξ)  = α * ξ
α_val = 0.3
B_valH(x, y)  = B_val(x,y,-H_val(x,y) )

# Define dictionaries for substitutions:
B_values     = Dict(
    B(x, y, ξ)=>B_val(x, y, ξ), 
    B(x, y, z)=>B_val(x, y, z), 
    B(x, y, -H(x,y))=>B_valH(x,y)
    )

Dict{Sym{PyCall.PyObject}, Sym{PyCall.PyObject}} with 3 entries:
  B(x, y, -H(x, y)) => α*(x^2 + y^2 - 1.0)
  B(x, y, ξ)        => α*ξ
  B(x, y, z)        => z*α

#### Compute buoyancy expression for Latex notes and check it:

In [11]:
Latex_b = ((- α * Γ)) / (ψ^2 * Ek) * ( 
(exp(ψ * H(x,y)) * (exp(ψ * z) - exp(-ψ * z))) /(ψ * (1 + exp(2 * ψ * H(x,y))))
- z
)
@assert b.subs(B_values) - Latex_b == 0
println("2.7 LaTeX b(x) buoyancy expression with linear buoyancy relaxation profile matches.")

2.7 LaTeX b(x) buoyancy expression with linear buoyancy relaxation profile matches.
